# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os, sys, subprocess

# IMPORTANT: never `import mani_skill` directly in this kernel for the health check.
# Jupyter's own logs showed repeated 'AsyncIOLoopKernelRestarter: restarting kernel'
# events — the kernel process itself was crashing (likely a native segfault in
# mani_skill's package init touching Vulkan/GPU init, not a catchable Python
# exception). Running the check in a throwaway subprocess means a crash there
# only kills the subprocess — this kernel survives and we get a clean exit code.
def _mani_skill_ok(timeout=60):
    r = subprocess.run([sys.executable, '-c', 'import mani_skill.utils'],
                        capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, r

ok, r = _mani_skill_ok()
if not ok:
    print(f'mani_skill broken (subprocess exit code {r.returncode}) — reinstalling...')
    if r.stderr.strip():
        print('Import check stderr (last 1500 chars):')
        print(r.stderr[-1500:])
    try:
        rr = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--force-reinstall',
             '--no-deps', '--no-cache-dir', 'mani-skill'],
            capture_output=True, text=True, timeout=180,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            'pip install timed out after 180s — likely a network issue reaching '
            'PyPI from this Colab VM.'
        )
    if rr.returncode != 0:
        print('Reinstall FAILED — pip output:')
        print(rr.stdout[-3000:])
        print(rr.stderr[-3000:])
        raise RuntimeError('mani-skill reinstall failed — see pip output above')

    ok2, r2 = _mani_skill_ok()
    if not ok2:
        print(f'Still broken after reinstall (subprocess exit code {r2.returncode}):')
        print(r2.stderr[-3000:])
        raise RuntimeError(
            'mani_skill still broken (or still segfaulting) after reinstall. '
            'This points to an environment-level problem with this specific VM '
            '(e.g. GPU/Vulkan driver state) rather than a corrupted package. '
            'Try: Runtime > Disconnect and delete runtime, then retry on a fresh VM.'
        )
    print('mani_skill verified OK in isolated subprocess. Continuing...')

if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.chdir("/content/rdt-igtesting")
os.system("git pull -q")

TASKS = ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]
for task in TASKS:
    shutil.copy(
        hf_hub_download("robotics-diffusion-transformer/maniskill-model",
                        "lang_embeds/text_embed_" + task + ".pt"),
        "lang_embed_" + task + ".pt"
    )
    print("Downloaded lang embed:", task)

shutil.copy("lang_embed_PickCube-v1.pt", "lang_embed.pt")
print("Ready.")

In [ ]:
# @title Rollout settings
n_successes = 1  # @param {type:"integer"}
task = "PickCube-v1"  # @param ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]

from run_rollout_cell import run
run(task=task, n=n_successes)

In [ ]:
import os
os.environ["SKIP_IG"] = "1"
os.environ["MANISKILL_TASK"] = task
os.environ["LANG_EMBED"] = "lang_embed_" + task + ".pt"
%run -i rdt_blurig.py
%run -i run_ig.py